# 02 — Reference Projection (R / Seurat CCA)

**Method:** Seurat Canonical Correlation Analysis (CCA) label transfer —
`FindTransferAnchors(reduction = 'cca')` + `TransferData` + `MapQuery`.

This is the **finalised pipeline method**, selected after benchmarking against scVI/scANVI
(see `notebooks/benchmarking/benchmark_celltype.ipynb`).

**Seurat CCA workflow:**
1. Build reference atlas (all TILs annotated, 1 patient left out split)
2. `FindTransferAnchors` — identifies shared biological states via CCA
3. `TransferData` — transfers cell-type labels to query
4. `MapQuery` — projects query onto reference UMAP
5. Evaluate transfer accuracy (demo only — ground truth available from split)
6. Identify CD8+ T cells for downstream clonotype analysis

**Papermill parameters:** `reference_path`, `query_path`

In [ ]:
# Papermill parameter cell
reference_path <- "data/cca/reference.rds"
query_path     <- "data/cca/query.rds"
output_path    <- "data/cca/query_projected.rds"
n_dims         <- 30L

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(patchwork)
  library(dplyr)
})
set.seed(42)
options(repr.plot.width = 9, repr.plot.height = 4)  # compact figures
cat(sprintf("Seurat %s\n", packageVersion("Seurat")))

.nb02_start <- proc.time()[[3]]
cat(sprintf("NB02 started at %s\n", format(Sys.time(), "%H:%M:%S")))

## 1 · Build Reference Atlas

In production: load a pre-built pan-cancer TIL reference atlas (`.rds`).  
Here we split `all TILs` (pre-annotated) (n-1)/1 to simulate a reference vs. new patient sample.

In [ ]:
build_reference <- function() {
  stop("build_reference() should not be called — reference.rds is built by 00_data_acquisition.ipynb")
}

if (file.exists(reference_path) && file.exists(query_path)) {
  cat("Loading pre-built reference and query from disk...\n")
  ref   <- readRDS(reference_path)
  query <- readRDS(query_path)
} else {
  stop("reference.rds or query.rds not found — run 00_data_acquisition.ipynb first")
}

cat(sprintf("Reference: %d cells | Query: %d cells\n", ncol(ref), ncol(query)))

# Determine annotation column (NB00 uses 'cluster'; uses 'annotation_col')
if ("cluster" %in% colnames(ref@meta.data)) {
  annotation_col <- "cluster"
} else if (annotation_col %in% colnames(ref@meta.data)) {
  annotation_col <- annotation_col
} else {
  # fallback: use first character metadata column
  char_cols <- names(Filter(is.character, as.data.frame(ref@meta.data)))
  annotation_col <- char_cols[1]
  warning(sprintf("No standard annotation column found; using '%s'", annotation_col))
}
cat(sprintf("Using '%s' as reference annotation column\n", annotation_col))
cat("Reference cell types:\n")
print(table(ref@meta.data[[annotation_col]]))


## 2 · Prepare Reference

Ensure the reference has PCA computed and UMAP fit with `return.model = TRUE` (required by `MapQuery`).

In [ ]:
# Re-run UMAP with return.model = TRUE if not already present
if (!"model" %in% names(ref@reductions$umap)) {
  cat("Running PCA + neighbours + UMAP on reference...\n")
  # If PCA not available, recompute
  if (!"pca" %in% names(ref@reductions)) {
    ref <- FindVariableFeatures(ref, nfeatures = 2000, verbose = FALSE)
    ref <- ScaleData(ref, features = VariableFeatures(ref), verbose = FALSE)
    ref <- RunPCA(ref, npcs = 50, verbose = FALSE)
  }
  ref <- FindNeighbors(ref, dims = 1:n_dims, verbose = FALSE)
  ref <- RunUMAP(ref, dims = 1:n_dims, return.model = TRUE, verbose = FALSE)
} else {
  cat("Reference already has UMAP model — reusing\n")
}

DimPlot(ref, reduction = "umap", group.by = annotation_col,
        label = TRUE, label.size = 3, pt.size = 0.5) +
  ggtitle("Reference atlas") + NoLegend()

## 3 · Find CCA Transfer Anchors

`FindTransferAnchors(reduction = 'cca')` identifies shared biological variation between reference and query by projecting both datasets into a common CCA space. Anchor pairs are high-confidence cell correspondences.

This mirrors Seurat's original CCA integration described in Stuart et al., Cell 2019.

In [ ]:
cat("Finding CCA transfer anchors...\n")
.anchors_start <- proc.time()[[3]]
anchors <- FindTransferAnchors(
  reference  = ref,
  query      = query,
  dims       = 1:n_dims,
  reduction           = "pcaproject", # PCA projection — O(n), much less memory than CCA
  reference.reduction = "pca",             # required for MapQuery
  verbose             = FALSE,
  approx.pca          = TRUE
)
cat(sprintf("Found %d anchors\n", nrow(anchors@anchors)))
.anchors_elapsed <- proc.time()[[3]] - .anchors_start
cat(sprintf("FindTransferAnchors elapsed: %.1f s (%.1f min)\n", .anchors_elapsed, .anchors_elapsed / 60))

## 4 · Transfer Cell-type Labels

`TransferData` uses the CCA anchors to compute a weighted prediction score for each cell-type label.

In [ ]:
predictions <- TransferData(
  anchorset        = anchors,
  refdata          = ref@meta.data[[annotation_col]],
  dims      = 1:n_dims
)
query <- AddMetaData(query, metadata = predictions)

cat("Transferred label distribution (query):\n")
print(table(query$predicted.id))

# Prediction score distribution
ggplot(query@meta.data, aes(x = prediction.score.max)) +
  geom_histogram(bins = 40, fill = "steelblue", colour = "white") +
  labs(x = "Max prediction score", y = "# cells",
       title = "CCA label transfer confidence scores") +
  theme_classic()

## 5 · MapQuery — Project onto Reference UMAP

`MapQuery` combines label transfer and dimensional reduction mapping, placing query cells into the reference UMAP space.

In [ ]:
query <- MapQuery(
  anchorset         = anchors,
  reference         = ref,
  query             = query,
  refdata           = list(celltype = annotation_col),
  reference.reduction = "pca",
  reduction.model   = "umap"
)

cat("Query embeddings after MapQuery:\n")
print(names(query@reductions))

In [ ]:
# Joint UMAP visualisation
ref$origin   <- "reference"
query$origin <- "query"

# Use ref.umap embedding to place both datasets
p_ref <- DimPlot(ref, reduction = "umap", group.by = annotation_col,
                 label = TRUE, label.size = 3, pt.size = 0.4) +
         NoLegend() + ggtitle("Reference")

p_qry <- DimPlot(query, reduction = "ref.umap", group.by = "predicted.id",
                 label = TRUE, label.size = 3, pt.size = 0.4) +
         NoLegend() + ggtitle("Query (projected)")

p_ref + p_qry

## 6 · Evaluate Label Transfer

Since we have ground-truth labels for the demo split, we can evaluate transfer accuracy.

In [ ]:
# Evaluation: ground-truth labels are stored under annotation_col ("cluster")
gt_col <- annotation_col

if (gt_col %in% colnames(query@meta.data)) {
  true_labels      <- query@meta.data[[gt_col]]
  predicted_labels <- query$predicted.id

  conf <- table(Predicted = predicted_labels, True = true_labels)
  cat("=== Confusion matrix (predicted rows x true columns) ===\n")
  print(conf)

  acc <- sum(diag(conf)) / sum(conf)
  cat(sprintf("\nOverall label transfer accuracy: %.1f%%\n", acc * 100))

  conf_df <- as.data.frame(conf)
  p_conf <- ggplot(conf_df, aes(x = True, y = Predicted, fill = Freq)) +
    geom_tile(colour = "grey90") +
    geom_text(aes(label = Freq), size = 3) +
    scale_fill_gradient(low = "white", high = "steelblue") +
    theme_classic() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(title = "CCA label transfer — confusion matrix",
         x = "True cell type", y = "Predicted cell type")
  print(p_conf)

  # ── CD8_ex precision / recall (CD8_ex = positive class) ────────────
  pos_class <- "CD8_ex"
  tp <- sum(predicted_labels == pos_class & true_labels == pos_class)
  fp <- sum(predicted_labels == pos_class & true_labels != pos_class)
  fn <- sum(predicted_labels != pos_class & true_labels == pos_class)
  prec <- if ((tp + fp) > 0) tp / (tp + fp) else NA_real_
  rec  <- if ((tp + fn) > 0) tp / (tp + fn) else NA_real_
  f1   <- if (!is.na(prec) && !is.na(rec) && (prec + rec) > 0)
            2 * prec * rec / (prec + rec)
          else NA_real_

  cat("\n=== CD8_ex (positive class) precision / recall ===\n")
  cat(sprintf("  True positives  (TP): %d\n", tp))
  cat(sprintf("  False positives (FP): %d\n", fp))
  cat(sprintf("  False negatives (FN): %d\n", fn))
  cat(sprintf("  Precision : %.3f\n", prec))
  cat(sprintf("  Recall    : %.3f\n", rec))
  cat(sprintf("  F1        : %.3f\n", f1))

} else {
  cat(sprintf("Ground-truth column '%s' not found in query — skipping evaluation.\n", gt_col))
}

## 7 · Identify CD8+ T Cells

In [ ]:
query$is_cd8 <- grepl("CD8", query$predicted.id, ignore.case = TRUE)
n_cd8 <- sum(query$is_cd8)
cat(sprintf("CD8+ T cells in query: %d / %d (%.1f%%)\n",
            n_cd8, ncol(query), n_cd8 / ncol(query) * 100))

DimPlot(query, reduction = "ref.umap",
        cells.highlight = colnames(query)[query$is_cd8],
        cols.highlight = "#e63946", cols = "lightgrey",
        pt.size = 0.5) +
  ggtitle(sprintf("CD8+ T cells highlighted (n = %d)", n_cd8))

## 8 · Save Projected Query

In [ ]:
saveRDS(query, file = output_path)
cat(sprintf("Saved projected Seurat object → %s\n", output_path))
cat(sprintf("Key metadata columns: %s\n",
            paste(colnames(query@meta.data), collapse = ", ")))

.nb02_elapsed <- proc.time()[[3]] - .nb02_start
cat(sprintf("\n=== NB02 Timing Summary ===\n"))
cat(sprintf("  FindTransferAnchors : %6.1f s (%5.1f min)\n", .anchors_elapsed, .anchors_elapsed / 60))
cat(sprintf("  Total NB02 elapsed  : %6.1f s (%5.1f min)\n", .nb02_elapsed, .nb02_elapsed / 60))